# LLM answer evaluation: Gold standard vs. Answer A and B

This notebook compares two candidate answers against one gold-standard answer using:

1. **ROUGE** — lexical overlap (ROUGE-1, ROUGE-2, ROUGE-L).
2. **Semantic similarity** — cosine similarity between sentence embeddings.
3. **Natural-language inference (NLI)** — entailment, contradiction, and neutral probabilities.

Edit the three text variables in the next cell, then run the notebook from top to bottom.

> **Interpretation note:** These metrics measure different properties and should be treated as complementary signals, not as a definitive or context-free score of correctness. In particular, NLI entailment is directional: here the gold standard is the *premise* and the candidate answer is the *hypothesis*.

In [1]:
# ── INPUTS: edit these three texts ───────────────────────────────────────

gold_standard = """
Yes, the current throughput of the system is 100 requests per second, which is sufficient based on the minimum requirement of 80 requests per second. The system is performing well under the current load and can handle the expected traffic without any issues.
"""

answer_a = """
The current throughput of the system is 100 requests per second, which exceeds the minimum requirement of 80 requests per second. Therefore, the system is performing well.
"""

answer_b = """
Yes, the system has a throughput of 150 requests per second and is performing well under the current load. The minimum requirement is 50 requests per second, so the system is more than capable of handling the expected traffic.
"""


## 0. Setup

Install the dependencies once in the notebook environment. The first model run may download model weights from Hugging Face.

In [2]:
%pip install -q rouge-score sentence-transformers transformers torch pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import torch
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Basic input validation
texts = {
    "Gold standard": gold_standard.strip(),
    "Answer A": answer_a.strip(),
    "Answer B": answer_b.strip(),
}
if not all(texts.values()):
    raise ValueError("Please provide non-empty text for the gold standard, Answer A, and Answer B.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 1. ROUGE

ROUGE measures n-gram and longest-common-subsequence overlap. The F1 score balances precision and recall; higher values indicate greater lexical overlap with the reference. It does not establish factual correctness.

In [4]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge_rows = []
for answer_name in ("Answer A", "Answer B"):
    scores = scorer.score(texts["Gold standard"], texts[answer_name])
    row = {"Candidate": answer_name}
    for metric, score in scores.items():
        row[f"{metric.upper()} precision"] = score.precision
        row[f"{metric.upper()} recall"] = score.recall
        row[f"{metric.upper()} F1"] = score.fmeasure
    rouge_rows.append(row)

rouge_results = pd.DataFrame(rouge_rows).set_index("Candidate")
display(rouge_results.round(4))

,ROUGE1 precision,ROUGE1 recall,ROUGE1 F1,ROUGE2 precision,ROUGE2 recall,ROUGE2 F1,ROUGEL precision,ROUGEL recall,ROUGEL F1
Candidate,,,,,,,,,
Answer A,0.9259,0.5814,0.7143,0.8462,0.5238,0.6471,0.9259,0.5814,0.7143
Answer B,0.7949,0.7209,0.7561,0.5263,0.4762,0.5000,0.5385,0.4884,0.5122


## 2. Semantic similarity (sentence embeddings)

This section uses `sentence-transformers/all-MiniLM-L6-v2`, a compact sentence-embedding model, and cosine similarity. This is generally more suitable for comparing sentence meaning than averaging static Word2Vec word vectors. Scores are cosine similarities (typically between -1 and 1); higher means the embeddings are closer, not necessarily that the answer is correct.

In [5]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
candidate_texts = [texts["Answer A"], texts["Answer B"]]

embeddings = embedding_model.encode(
    [texts["Gold standard"]] + candidate_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)

# With normalized embeddings, dot product equals cosine similarity.
similarities = torch.matmul(embeddings[1:], embeddings[0]).cpu().numpy()

semantic_results = pd.DataFrame({
    "Candidate": ["Answer A", "Answer B"],
    "Cosine similarity": similarities
}).set_index("Candidate")

display(semantic_results.round(4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1409.67it/s]


,Cosine similarity
Candidate,
Answer A,0.8368
Answer B,0.9256


## 3. Entailment / contradiction (DeBERTa NLI)

The model `cross-encoder/nli-deberta-v3-base` returns probabilities for **contradiction**, **entailment**, and **neutral**. The pair is ordered as:

- **Premise:** gold-standard text
- **Hypothesis:** candidate answer

A high entailment probability indicates that the candidate is supported by the gold standard. A high contradiction probability is a warning that the candidate conflicts with it. Neutral means the premise neither supports nor contradicts the hypothesis. Label IDs are read from the model configuration rather than assumed.

In [6]:
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
nli_model.eval()

# Normalize model labels so the output is robust to label-ID ordering.
id2label = {
    int(i): str(label).lower()
    for i, label in nli_model.config.id2label.items()
}
print("Model labels:", id2label)

def canonical_nli_label(label):
    label = label.lower()
    if "entail" in label:
        return "entailment"
    if "contrad" in label:
        return "contradiction"
    if "neutral" in label:
        return "neutral"
    return label

def get_nli_probabilities(premise, hypothesis):
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1).squeeze(0).cpu().tolist()

    result = {canonical_nli_label(id2label[i]): float(probabilities[i])
              for i in range(len(probabilities))}
    return result

nli_rows = []
for answer_name in ("Answer A", "Answer B"):
    probs = get_nli_probabilities(texts["Gold standard"], texts[answer_name])
    nli_rows.append({
        "Candidate": answer_name,
        "Entailment probability": probs.get("entailment", float("nan")),
        "Contradiction probability": probs.get("contradiction", float("nan")),
        "Neutral probability": probs.get("neutral", float("nan")),
    })

nli_results = pd.DataFrame(nli_rows).set_index("Candidate")
display(nli_results.round(4))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 391.60it/s]


Model labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


,Entailment probability,Contradiction probability,Neutral probability
Candidate,,,
Answer A,0.8854,0.0004,0.1143
Answer B,0.0001,0.9996,0.0003


## 4. Side-by-side summary

The table below puts the principal comparison values together. ROUGE-L F1 is shown as a compact lexical-overlap indicator; semantic similarity and NLI probabilities are shown separately because they are not interchangeable.

In [7]:
summary = pd.DataFrame(index=["Answer A", "Answer B"])
summary["ROUGE-1 F1"] = rouge_results["ROUGE1 F1"]
summary["ROUGE-2 F1"] = rouge_results["ROUGE2 F1"]
summary["ROUGE-L F1"] = rouge_results["ROUGEL F1"]
summary["Embedding cosine similarity"] = semantic_results["Cosine similarity"]
summary["NLI entailment"] = nli_results["Entailment probability"]
summary["NLI contradiction"] = nli_results["Contradiction probability"]
summary["NLI neutral"] = nli_results["Neutral probability"]

display(summary.round(4))

,ROUGE-1 F1,ROUGE-2 F1,ROUGE-L F1,Embedding cosine similarity,NLI entailment,NLI contradiction,NLI neutral
Answer A,0.7143,0.6471,0.7143,0.8368,0.8854,0.0004,0.1143
Answer B,0.7561,0.5000,0.5122,0.9256,0.0001,0.9996,0.0003


## Notes on interpretation and limitations

- **ROUGE:** sensitive to wording and length; a correct paraphrase can receive a modest score.
- **Embedding similarity:** captures semantic proximity, but similar-sounding statements can still differ in key facts (for example, names, dates, or quantities).
- **NLI:** is directional and model-dependent. Reversing premise and hypothesis can change the result. Long inputs may be truncated to the model's maximum sequence length.
- **No single metric is a ground-truth correctness score.** For high-stakes evaluation, add task-specific checks (for example, exact-match checks for numbers/entities), human review, and a representative validation set.

The displayed values compare each candidate independently against the gold standard; they do not compare Answer A directly with Answer B.